
# FADID-PV: Frequency-Aware Deep Information Decomposition with Physiological Verification
### A generalizable deepfake-image detection pipeline

This notebook implements a working, trainable version of the **FADID-PV** architecture
described in *"Advancements in Deepfake Image Detection: Bridging Research Gaps and
Navigating Patentability"* (2026 report). It is built to run end-to-end in Google Colab
or a local Jupyter environment with a CUDA GPU (CPU also works for small smoke tests).

**Design goal:** don't memorize the dataset — generalize.

Every design choice below exists specifically to fight the four research gaps identified
in the report:

| Research gap (from the report) | What this notebook does about it |
|---|---|
| Cross-dataset generalization deficit | Frozen foundation-model backbone (only LayerNorm + head trained, ~0.03% of params, GenD-style) so natural-image priors aren't destroyed; adversarial **Deep Information Decomposition (DID)** strips generator/identity-specific signal out of the forensic feature |
| Vulnerability to degradation & adversarial perturbation | **Extreme Compound Degradation Engine** — JPEG cycling, blur, noise, resolution changes — applied *during training*, not just at test time |
| Spatial shortcut learning | **Frequency-domain masking** (FFT band-dropping) forces the network off easy spatial blending-edge cues and onto invariant structure |
| Physiological/physical blind spot | **Corneal reflection consistency stream** — an orthogonal signal generators rarely get right, independent of pixel-level artifacts |

We also bake in an explicit **anti-overfitting evaluation protocol**: splits are done by
`source_video_id` / `identity_id` / `generator` — never by random image shuffling — and we
include a **leave-one-generator-out ("zero-day") test** as the headline generalization metric,
not just intra-dataset accuracy.

> ⚠️ This is a research/education scaffold, not a production system. Cells that download
> pretrained backbone weights need internet access (Colab has this by default). Detecting
> real people's faces raises real privacy/consent obligations — see the ethics note at the
> bottom of the notebook before pointing this at anything other than a licensed research dataset.


## 1. Environment setup

Run this once. `timm` gives us pretrained DINOv2 / CLIP ViT backbones without extra
plumbing; `mediapipe` gives us iris landmarks for the physiological stream for free
(478-point face mesh, indices 468-477 are the iris); `albumentations` implements the
degradation engine efficiently.


In [ ]:

# !pip install -q torch torchvision timm mediapipe albumentations opencv-python-headless \
#     scikit-learn pandas matplotlib tqdm

import os, io, math, json, random, glob
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 2. Configuration

All hyperparameters in one place. `alpha/beta/gamma` are the composite-loss weights from
the report's

$$\mathcal{L}_{total} = \mathcal{L}_{cls} + \alpha\,\mathcal{L}_{mi} + \beta\,\mathcal{L}_{physio} + \gamma\,\mathcal{L}_{metric}$$


In [ ]:

@dataclass
class Config:
    # data
    manifest_csv: str = "manifest.csv"          # see section 3 for required schema
    image_size: int = 224
    batch_size: int = 16
    num_workers: int = 4
    val_fraction: float = 0.15
    test_fraction: float = 0.15
    holdout_generator: Optional[str] = None      # e.g. "StyleGAN3" -> excluded from train/val entirely
    split_key: str = "source_video_id"           # never split on raw image / random index

    # backbone
    backbone_name: str = "vit_large_patch14_dinov2.lvd142m"  # timm name; swap for a CLIP ViT-L/14 variant if preferred
    backbone_img_size: int = 224
    freeze_backbone: bool = True                 # only LayerNorm (+ head) trainable (GenD-style)
    embed_dim: int = 1024

    # frequency masking
    freq_mask_prob: float = 0.5
    freq_bands: Tuple[str, ...] = ("low", "mid", "high")

    # DID module
    did_hidden: int = 512
    did_out: int = 256
    num_nuisance_classes: int = 8                # generator/technique id cardinality in your manifest
    grl_lambda: float = 1.0

    # physiological stream
    eye_crop_size: int = 64
    physio_hidden: int = 128

    # fusion / metric head
    fusion_dim: int = 256

    # loss weights
    alpha_mi: float = 0.3
    beta_physio: float = 0.2
    gamma_metric: float = 0.5
    metric_temperature: float = 0.07

    # optimisation
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 20
    amp: bool = True

    # degradation engine
    degradation_prob: float = 0.7

cfg = Config()
cfg


## 3. Data manifest & the anti-leakage split

**This is the single most important cell for generalization.** Random image-level splits
leak identity/compression/source-video information between train and test and are the
single biggest reason published deepfake detectors look great on paper and collapse in
the wild (see "Cross-Dataset Generalization Deficit" in the report).

Your manifest CSV needs these columns:

| column | meaning |
|---|---|
| `image_path` | path to the image file |
| `label` | `0` = real, `1` = fake |
| `source_video_id` | id shared by a real clip and all its fake derivatives — **split on this, never on image_path** |
| `generator` | e.g. `FaceSwap`, `StyleGAN3`, `DiffusionFaceSwap`, `real` for reals — used both as the DID nuisance label and for the leave-one-generator-out test |
| `identity_id` | (optional) subject identity, used as a second nuisance signal for the DID discriminator |
| `dataset_name` | (optional) which benchmark this row came from (FF++, Celeb-DF, DFDC, DF-Wild, ...) — used to report cross-dataset AUC separately per dataset |

If you don't have this yet, build it from FaceForensics++ / Celeb-DF / DFDC / DF-Wild style
folder layouts with a short script that walks the directories and fills these columns —
happy to write that for you once I know which datasets you're using.


In [ ]:

def load_manifest(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"image_path", "label", "source_video_id", "generator"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"manifest is missing required columns: {missing}")
    if "identity_id" not in df.columns:
        df["identity_id"] = df["source_video_id"]
    if "dataset_name" not in df.columns:
        df["dataset_name"] = "default"
    return df


def group_split(df: pd.DataFrame, cfg: Config):
    '''Split by group key (source video), never by row, so no real/fake pair or
    near-duplicate frame ever crosses the train/val/test boundary.'''
    groups = df[cfg.split_key].unique()
    rng = np.random.RandomState(SEED)
    rng.shuffle(groups)

    n = len(groups)
    n_test = max(1, int(n * cfg.test_fraction))
    n_val = max(1, int(n * cfg.val_fraction))

    test_groups = set(groups[:n_test])
    val_groups = set(groups[n_test:n_test + n_val])
    train_groups = set(groups[n_test + n_val:])

    train_df = df[df[cfg.split_key].isin(train_groups)].reset_index(drop=True)
    val_df = df[df[cfg.split_key].isin(val_groups)].reset_index(drop=True)
    test_df = df[df[cfg.split_key].isin(test_groups)].reset_index(drop=True)

    # Zero-day generalization: optionally strip one whole generator out of train/val
    # and evaluate on it separately -- this is the headline "does it generalize to an
    # unseen technique" number, closer to the NTIRE 2026 robustness framing than plain AUC.
    holdout_df = None
    if cfg.holdout_generator is not None:
        mask_train = train_df["generator"] == cfg.holdout_generator
        mask_val = val_df["generator"] == cfg.holdout_generator
        holdout_df = pd.concat([train_df[mask_train], val_df[mask_val]], ignore_index=True)
        train_df = train_df[~mask_train].reset_index(drop=True)
        val_df = val_df[~mask_val].reset_index(drop=True)
        test_df = pd.concat([test_df, holdout_df], ignore_index=True)  # unseen generator only ever appears in test

    return train_df, val_df, test_df


# Example (uncomment once manifest.csv exists):
# df = load_manifest(cfg.manifest_csv)
# train_df, val_df, test_df = group_split(df, cfg)
# print(len(train_df), len(val_df), len(test_df))


## 4. Frequency-domain masking

Implements the report's spectral masking exactly:
$X=\mathcal{F}(x)$, $\tilde X(f) = M(f)\cdot X(f)$, $\tilde x = \mathcal{F}^{-1}(\tilde X)$,
with the low/mid/high band definitions from the "Frequency-Domain Masking and Spectral
Representation" section. Randomly zeroing a band during training prevents the network from
memorizing a specific generator's upsampling noise signature.


In [ ]:

def _band_mask(h: int, w: int, band: str) -> np.ndarray:
    mask = np.ones((h, w), dtype=np.float32)
    u0, u1 = 0, h
    v0, v1 = 0, w
    if band == "low":
        u0, u1 = 0, h // 4
        v0, v1 = 0, w // 4
    elif band == "mid":
        u0, u1 = h // 4, (3 * h) // 4
        v0, v1 = w // 4, (3 * w) // 4
    elif band == "high":
        u0, u1 = (3 * h) // 4, h
        v0, v1 = (3 * w) // 4, w
    else:
        raise ValueError(band)
    mask[u0:u1, v0:v1] = 0.0
    return mask


def frequency_domain_mask(img: np.ndarray, band: str) -> np.ndarray:
    '''img: HxWxC float32 in [0,1]. Zeroes the requested spectral band per channel.'''
    h, w = img.shape[:2]
    mask = _band_mask(h, w, band)
    out = np.empty_like(img)
    for c in range(img.shape[2]):
        f = np.fft.fftshift(np.fft.fft2(img[..., c]))
        f *= mask
        out[..., c] = np.real(np.fft.ifft2(np.fft.ifftshift(f)))
    return np.clip(out, 0.0, 1.0)


def maybe_apply_frequency_mask(img_float01: np.ndarray, cfg: Config) -> np.ndarray:
    if random.random() > cfg.freq_mask_prob:
        return img_float01
    band = random.choice(cfg.freq_bands)
    return frequency_domain_mask(img_float01, band)


## 5. Extreme Compound Degradation Engine

Simulates the real-world degradations from the report's robustness discussion
(cyclic JPEG compression, resolution changes, blur, salt-and-pepper noise, simple
H.264-style blockiness) applied *stochastically during training* so the model never
gets to rely on pristine high-frequency artifacts.


In [ ]:

def cyclic_jpeg(img_bgr: np.ndarray, cycles: int = 2, quality_range=(25, 70)) -> np.ndarray:
    out = img_bgr.copy()
    for _ in range(cycles):
        q = random.randint(*quality_range)
        ok, enc = cv2.imencode(".jpg", out, [int(cv2.IMWRITE_JPEG_QUALITY), q])
        if ok:
            out = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return out


def resolution_roundtrip(img_bgr: np.ndarray, min_scale=0.3, max_scale=0.8) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    scale = random.uniform(min_scale, max_scale)
    small = cv2.resize(img_bgr, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_LINEAR)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def salt_and_pepper(img_bgr: np.ndarray, amount=0.01) -> np.ndarray:
    out = img_bgr.copy()
    h, w = out.shape[:2]
    n = int(amount * h * w)
    ys = np.random.randint(0, h, n); xs = np.random.randint(0, w, n)
    out[ys[: n // 2], xs[: n // 2]] = 255
    out[ys[n // 2:], xs[n // 2:]] = 0
    return out


def gaussian_blur(img_bgr: np.ndarray) -> np.ndarray:
    k = random.choice([3, 5, 7])
    return cv2.GaussianBlur(img_bgr, (k, k), 0)


def block_loss_sim(img_bgr: np.ndarray, block=16, drop_prob=0.03) -> np.ndarray:
    '''Crude stand-in for H.264 packet-loss macroblock corruption.'''
    out = img_bgr.copy()
    h, w = out.shape[:2]
    for y in range(0, h, block):
        for x in range(0, w, block):
            if random.random() < drop_prob:
                out[y:y + block, x:x + block] = np.mean(out[y:y + block, x:x + block], axis=(0, 1), keepdims=True)
    return out


DEGRADATIONS = [cyclic_jpeg, resolution_roundtrip, salt_and_pepper, gaussian_blur, block_loss_sim]


def apply_compound_degradation(img_bgr: np.ndarray, cfg: Config, max_ops: int = 3) -> np.ndarray:
    if random.random() > cfg.degradation_prob:
        return img_bgr
    n_ops = random.randint(1, max_ops)
    ops = random.sample(DEGRADATIONS, n_ops)
    out = img_bgr
    for op in ops:
        out = op(out)
    return out


## 6. Face alignment & iris landmark extraction

RetinaFace-quality alignment plus MediaPipe's refined 478-point face mesh (iris points
468-477) gives us, in one pass: (a) an aligned face crop with a 1.3x margin as in the
report's pre-processing stage, and (b) left/right eye crops for the corneal-reflection
consistency stream.


In [ ]:

import mediapipe as mp

_mp_face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1, refine_landmarks=True,
    min_detection_confidence=0.5,
)

LEFT_IRIS = [469, 470, 471, 472]
RIGHT_IRIS = [474, 475, 476, 477]


def align_and_extract(img_bgr: np.ndarray, out_size: int, margin: float = 1.3):
    '''Returns (aligned_face_bgr, left_eye_crop, right_eye_crop) or (None, None, None)
    if no face is found.'''
    h, w = img_bgr.shape[:2]
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = _mp_face_mesh.process(rgb)
    if not res.multi_face_landmarks:
        return None, None, None

    lm = res.multi_face_landmarks[0].landmark
    pts = np.array([[p.x * w, p.y * h] for p in lm])

    x0, y0 = pts[:, 0].min(), pts[:, 1].min()
    x1, y1 = pts[:, 0].max(), pts[:, 1].max()
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    bw, bh = (x1 - x0) * margin, (y1 - y0) * margin
    x0, x1 = int(max(0, cx - bw / 2)), int(min(w, cx + bw / 2))
    y0, y1 = int(max(0, cy - bh / 2)), int(min(h, cy + bh / 2))
    face = img_bgr[y0:y1, x0:x1]
    if face.size == 0:
        return None, None, None
    face = cv2.resize(face, (out_size, out_size))

    def eye_crop(idx_list, size=64):
        eye_pts = pts[idx_list]
        ex, ey = eye_pts[:, 0].mean(), eye_pts[:, 1].mean()
        r = max(eye_pts[:, 0].max() - eye_pts[:, 0].min(), eye_pts[:, 1].max() - eye_pts[:, 1].min())
        r = max(r * 2.0, 8)
        ex0, ex1 = int(max(0, ex - r)), int(min(w, ex + r))
        ey0, ey1 = int(max(0, ey - r)), int(min(h, ey + r))
        crop = img_bgr[ey0:ey1, ex0:ex1]
        if crop.size == 0:
            return np.zeros((size, size, 3), dtype=np.uint8)
        return cv2.resize(crop, (size, size))

    left_eye = eye_crop(LEFT_IRIS)
    right_eye = eye_crop(RIGHT_IRIS)
    return face, left_eye, right_eye


## 7. `Dataset` class

Ties sections 3-6 together: loads an image, aligns the face, extracts eye crops, applies
the degradation engine + frequency masking as **training-time augmentation only**, and
returns everything the model needs (spatial tensor, frequency-masked tensor, eye crops,
label, nuisance ids for the DID discriminator).


In [ ]:

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def to_tensor_norm(img_bgr_uint8: np.ndarray) -> torch.Tensor:
    img = cv2.cvtColor(img_bgr_uint8, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(img.transpose(2, 0, 1)).float()


class DeepfakeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: Config, train: bool,
                 generator_to_id: Dict[str, int], identity_to_id: Dict[str, int]):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.train = train
        self.generator_to_id = generator_to_id
        self.identity_to_id = identity_to_id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["image_path"])
        if img is None:
            img = np.zeros((self.cfg.image_size, self.cfg.image_size, 3), dtype=np.uint8)

        face, left_eye, right_eye = align_and_extract(img, self.cfg.image_size)
        if face is None:
            face = cv2.resize(img, (self.cfg.image_size, self.cfg.image_size))
            left_eye = np.zeros((self.cfg.eye_crop_size, self.cfg.eye_crop_size, 3), dtype=np.uint8)
            right_eye = left_eye.copy()

        if self.train:
            face = apply_compound_degradation(face, self.cfg)

        face_float = cv2.cvtColor(face, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        freq_masked = maybe_apply_frequency_mask(face_float, self.cfg) if self.train else face_float
        freq_masked_bgr = cv2.cvtColor((freq_masked * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

        spatial_t = to_tensor_norm(face)
        freq_t = to_tensor_norm(freq_masked_bgr)
        left_t = to_tensor_norm(cv2.resize(left_eye, (self.cfg.eye_crop_size, self.cfg.eye_crop_size)))
        right_t = to_tensor_norm(cv2.resize(right_eye, (self.cfg.eye_crop_size, self.cfg.eye_crop_size)))

        label = torch.tensor(float(row["label"]))
        gen_id = torch.tensor(self.generator_to_id.get(row["generator"], 0), dtype=torch.long)
        ident_id = torch.tensor(self.identity_to_id.get(row["identity_id"], 0), dtype=torch.long)

        return {
            "spatial": spatial_t, "freq": freq_t,
            "left_eye": left_t, "right_eye": right_t,
            "label": label, "generator_id": gen_id, "identity_id": ident_id,
        }


def build_id_maps(df: pd.DataFrame):
    generators = sorted(df["generator"].unique().tolist())
    identities = sorted(df["identity_id"].unique().tolist())
    return {g: i for i, g in enumerate(generators)}, {p: i for i, p in enumerate(identities)}


## 8. Model architecture

### 8.1 Frozen foundation-model backbone (GenD-style parameter-efficient adaptation)

Only `LayerNorm` parameters are left trainable — everything else in the pretrained ViT
stays frozen, which is what lets the network keep its broad, generator-agnostic visual
prior instead of overfitting to whatever generator happens to dominate the training set.


In [ ]:

import timm

class FrozenLNBackbone(nn.Module):
    '''Wraps a timm ViT (DINOv2 / CLIP-style) backbone. Freezes everything except
    LayerNorm weights/biases and returns the CLS-token embedding.'''

    def __init__(self, model_name: str, img_size: int, embed_dim: int):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=img_size)
        self.embed_dim = embed_dim
        self._freeze_all_but_layernorm()

    def _freeze_all_but_layernorm(self):
        for name, p in self.backbone.named_parameters():
            p.requires_grad = ("norm" in name.lower()) and (
                name.lower().endswith(".weight") or name.lower().endswith(".bias")
            )
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.backbone.parameters())
        print(f"Backbone trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")

    def forward(self, x):
        return self.backbone(x)  # (B, embed_dim)


### 8.2 Gradient Reversal Layer + Deep Information Decomposition (DID) module

Splits the fused backbone embedding into a **forensic** branch (what actually indicates
"fake") and an **irrelevant** branch (identity/technique-specific nuisance). A
discriminator tries to guess the generator/identity from the forensic branch; the GRL
flips its gradient, so training *penalizes* the forensic encoder for leaking nuisance
information — forcing it toward purified, generator-agnostic features, exactly as in the
report's CrossDF-style adversarial minimax formulation.


In [ ]:

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)


class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, p_drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class DIDModule(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.forensic_encoder = MLP(cfg.embed_dim, cfg.did_hidden, cfg.did_out)
        self.irrelevant_encoder = MLP(cfg.embed_dim, cfg.did_hidden, cfg.did_out)
        self.grl_lambda = cfg.grl_lambda
        self.gen_discriminator = nn.Linear(cfg.did_out, cfg.num_nuisance_classes)
        self.id_discriminator = nn.Linear(cfg.did_out, cfg.num_nuisance_classes)

    def forward(self, embed):
        forensic = self.forensic_encoder(embed)
        irrelevant = self.irrelevant_encoder(embed)

        reversed_forensic = grad_reverse(forensic, self.grl_lambda)
        gen_logits = self.gen_discriminator(reversed_forensic)
        id_logits = self.id_discriminator(reversed_forensic)

        return forensic, irrelevant, gen_logits, id_logits


### 8.3 Physiological anchoring stream (corneal reflection consistency)

A small Residual-Attention-style CNN compares the left/right eye crops and outputs a
scalar anomaly score. This is intentionally independent of the frequency/spatial streams
above — a generator that has perfected spatial blending and spectral statistics still has
to get 3-D-consistent specular highlights right, which is a much harder, physics-grounded
constraint.


In [ ]:

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1), nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(self.avg(x))


class ResidualAttentionBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c),
        )
        self.attn = ChannelAttention(out_c)
        self.skip = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out = self.attn(out)
        return F.relu(out + self.skip(x))


class PhysioNet(nn.Module):
    '''Takes stacked left/right eye crops (B,6,H,W) -> scalar corneal-reflection
    anomaly score in [0,1] and a feature vector for fusion.'''

    def __init__(self, cfg: Config):
        super().__init__()
        self.stem = nn.Sequential(
            ResidualAttentionBlock(6, 32), nn.MaxPool2d(2),
            ResidualAttentionBlock(32, 64), nn.MaxPool2d(2),
            ResidualAttentionBlock(64, cfg.physio_hidden), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(cfg.physio_hidden, 1)
        self.feat_dim = cfg.physio_hidden

    def forward(self, left_eye, right_eye):
        x = torch.cat([left_eye, right_eye], dim=1)  # (B,6,H,W)
        feat = self.stem(x).flatten(1)
        anomaly_score = torch.sigmoid(self.head(feat))
        return anomaly_score.squeeze(-1), feat


### 8.4 Full FADID-PV model: fusion + hyperspherical classifier

Combines the two backbone passes (raw spatial + frequency-masked, following the report's
dual-stream description), the DID forensic token, and the physio stream feature into one
fused vector, L2-normalizes it onto a hypersphere, and classifies.


In [ ]:

class FADID_PV(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.spatial_backbone = FrozenLNBackbone(cfg.backbone_name, cfg.backbone_img_size, cfg.embed_dim)
        self.freq_backbone = FrozenLNBackbone(cfg.backbone_name, cfg.backbone_img_size, cfg.embed_dim)
        self.did = DIDModule(cfg)
        self.physio = PhysioNet(cfg)

        fused_in = cfg.did_out + cfg.physio_hidden
        self.fusion = MLP(fused_in, cfg.fusion_dim, cfg.fusion_dim)
        self.classifier = nn.Linear(cfg.fusion_dim, 1)

    def forward(self, batch):
        spatial_embed = self.spatial_backbone(batch["spatial"])
        freq_embed = self.freq_backbone(batch["freq"])
        combined_embed = spatial_embed + freq_embed  # simple, robust fusion of the two streams

        forensic, irrelevant, gen_logits, id_logits = self.did(combined_embed)
        physio_score, physio_feat = self.physio(batch["left_eye"], batch["right_eye"])

        fused = torch.cat([forensic, physio_feat], dim=1)
        fused = self.fusion(fused)
        fused_norm = F.normalize(fused, dim=1)  # hyperspherical projection

        logits = self.classifier(fused_norm).squeeze(-1)

        return {
            "logits": logits, "physio_score": physio_score,
            "gen_logits": gen_logits, "id_logits": id_logits,
            "fused_embed": fused_norm, "forensic": forensic, "irrelevant": irrelevant,
        }


## 9. Composite loss

Implements $\mathcal{L}_{total} = \mathcal{L}_{cls} + \alpha\mathcal{L}_{mi} + \beta\mathcal{L}_{physio} + \gamma\mathcal{L}_{metric}$ term-for-term.

* $\mathcal{L}_{cls}$ — BCE on the final real/fake logit.
* $\mathcal{L}_{mi}$ — cross-entropy of the GRL-reversed discriminators against generator/identity id (the reversal turns "minimize this" into "maximize the encoder's ability to fool it").
* $\mathcal{L}_{physio}$ — BCE of the physio anomaly score against the real/fake label (supervises the eye-consistency branch directly; swap in a purely geometric/self-supervised version if you have per-frame reflection-geometry ground truth).
* $\mathcal{L}_{metric}$ — supervised contrastive (SupCon-style) loss on the L2-normalized fused embedding: pulls same-label points together, pushes different-label points apart, independent of which generator produced them — this is what gives zero-shot separation for unseen generators.


In [ ]:

def supervised_contrastive_loss(embeddings: torch.Tensor, labels: torch.Tensor, temperature: float) -> torch.Tensor:
    device = embeddings.device
    labels = labels.view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)
    logits = torch.matmul(embeddings, embeddings.T) / temperature
    logits = logits - logits.max(dim=1, keepdim=True)[0].detach()  # numerical stability

    exp_logits = torch.exp(logits) * (1 - torch.eye(embeddings.size(0), device=device))
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)

    mask_sum = mask.sum(1) - 1  # exclude self
    mask_sum = torch.clamp(mask_sum, min=1)
    mean_log_prob_pos = (mask * log_prob).sum(1) / mask_sum - torch.diagonal(log_prob)  # remove self term contribution
    return -mean_log_prob_pos.mean()


def composite_loss(outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor], cfg: Config) -> Dict[str, torch.Tensor]:
    labels = batch["label"].to(outputs["logits"].device)

    cls_loss = F.binary_cross_entropy_with_logits(outputs["logits"], labels)

    mi_loss = F.cross_entropy(outputs["gen_logits"], batch["generator_id"].to(outputs["logits"].device)) + \
              F.cross_entropy(outputs["id_logits"], batch["identity_id"].to(outputs["logits"].device))

    physio_loss = F.binary_cross_entropy(outputs["physio_score"].clamp(1e-6, 1 - 1e-6), labels)

    metric_loss = supervised_contrastive_loss(outputs["fused_embed"], labels, cfg.metric_temperature)

    total = cls_loss + cfg.alpha_mi * mi_loss + cfg.beta_physio * physio_loss + cfg.gamma_metric * metric_loss

    return {
        "total": total, "cls": cls_loss, "mi": mi_loss,
        "physio": physio_loss, "metric": metric_loss,
    }


## 10. Training loop

Only the LayerNorm-unfrozen backbone params + DID module + physio net + fusion/classifier
head are ever passed to the optimizer — this is what keeps the trainable parameter count
tiny and prevents the catastrophic-forgetting-driven overfitting the report warns about.


In [ ]:

def build_optimizer(model: FADID_PV, cfg: Config):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Optimizing {n_trainable:,} / {n_total:,} params ({100*n_trainable/n_total:.2f}%)")
    return torch.optim.AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)


def move_batch(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def train_one_epoch(model, loader, optimizer, scaler, cfg, device):
    model.train()
    running = {"total": 0.0, "cls": 0.0, "mi": 0.0, "physio": 0.0, "metric": 0.0}
    n = 0
    for batch in loader:
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == "cuda"):
            outputs = model(batch)
            losses = composite_loss(outputs, batch, cfg)
        if scaler is not None:
            scaler.scale(losses["total"]).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            losses["total"].backward()
            optimizer.step()

        bs = batch["label"].size(0)
        for k in running:
            running[k] += losses[k].item() * bs
        n += bs
    return {k: v / max(n, 1) for k, v in running.items()}


@torch.no_grad()
def evaluate(model, loader, device, cfg) -> Dict[str, float]:
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        batch = move_batch(batch, device)
        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type == "cuda"):
            outputs = model(batch)
        all_logits.append(torch.sigmoid(outputs["logits"]).float().cpu().numpy())
        all_labels.append(batch["label"].cpu().numpy())
    probs = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan")

    fpr, tpr, _ = roc_curve(labels, probs) if len(np.unique(labels)) > 1 else (None, None, None)
    eer = float("nan")
    if fpr is not None:
        fnr = 1 - tpr
        idx = np.nanargmin(np.abs(fnr - fpr))
        eer = (fpr[idx] + fnr[idx]) / 2
    acc = ((probs > 0.5).astype(int) == labels).mean()
    return {"auc": auc, "eer": eer, "acc": acc}


In [ ]:

def run_training(cfg: Config, train_df, val_df, test_df):
    generator_to_id, identity_to_id = build_id_maps(pd.concat([train_df, val_df, test_df]))
    cfg.num_nuisance_classes = max(len(generator_to_id), len(identity_to_id))

    train_ds = DeepfakeDataset(train_df, cfg, train=True, generator_to_id=generator_to_id, identity_to_id=identity_to_id)
    val_ds = DeepfakeDataset(val_df, cfg, train=False, generator_to_id=generator_to_id, identity_to_id=identity_to_id)
    test_ds = DeepfakeDataset(test_df, cfg, train=False, generator_to_id=generator_to_id, identity_to_id=identity_to_id)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    model = FADID_PV(cfg).to(DEVICE)
    optimizer = build_optimizer(model, cfg)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp and DEVICE.type == "cuda")

    history = []
    best_val_auc = -1.0
    for epoch in range(cfg.epochs):
        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, cfg, DEVICE)
        val_metrics = evaluate(model, val_loader, DEVICE, cfg)
        history.append({"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}})
        print(f"[epoch {epoch}] train_loss={train_metrics['total']:.4f}  val_auc={val_metrics['auc']:.4f}  val_eer={val_metrics['eer']:.4f}")

        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            torch.save(model.state_dict(), "fadid_pv_best.pt")

    test_metrics = evaluate(model, test_loader, DEVICE, cfg)
    print("TEST (in-distribution + any held-out generator rows):", test_metrics)
    return model, pd.DataFrame(history), test_metrics


# Example run (uncomment once manifest.csv + Section 3 splits are ready):
# df = load_manifest(cfg.manifest_csv)
# train_df, val_df, test_df = group_split(df, cfg)
# model, history_df, test_metrics = run_training(cfg, train_df, val_df, test_df)


## 11. Generalization-focused evaluation

Don't trust a single overall AUC. Report it broken out by:

1. **Per-dataset AUC** (cross-dataset generalization — the FF++→Celeb-DF style table from the report).
2. **Zero-day generator AUC** — performance on `cfg.holdout_generator`, a technique the model never trained on at all.
3. **Robustness curve** — AUC as a function of degradation severity, mirroring the NTIRE 2026 challenge protocol.


In [ ]:

@torch.no_grad()
def evaluate_by_group(model, df: pd.DataFrame, cfg: Config, group_col: str, generator_to_id, identity_to_id, device):
    results = {}
    for group_val, sub_df in df.groupby(group_col):
        if sub_df["label"].nunique() < 2:
            continue
        ds = DeepfakeDataset(sub_df, cfg, train=False, generator_to_id=generator_to_id, identity_to_id=identity_to_id)
        loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
        results[group_val] = evaluate(model, loader, device, cfg)
    return pd.DataFrame(results).T


def plot_robustness_curve(model, base_df: pd.DataFrame, cfg: Config, generator_to_id, identity_to_id, device,
                           severities=(0.0, 0.3, 0.5, 0.7, 0.9)):
    '''Sweeps cfg.degradation_prob at eval time to trace an AUC-vs-degradation curve.'''
    aucs = []
    for sev in severities:
        local_cfg = Config(**{**cfg.__dict__, "degradation_prob": sev})
        ds = DeepfakeDataset(base_df, local_cfg, train=True, generator_to_id=generator_to_id, identity_to_id=identity_to_id)
        loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
        m = evaluate(model, loader, device, cfg)
        aucs.append(m["auc"])

    plt.figure()
    plt.plot(severities, aucs, marker="o")
    plt.xlabel("Degradation probability at eval time")
    plt.ylabel("AUC")
    plt.title("Robustness under increasing degradation")
    plt.ylim(0.4, 1.0)
    plt.grid(True)
    plt.show()
    return dict(zip(severities, aucs))


# Example:
# by_dataset = evaluate_by_group(model, test_df, cfg, "dataset_name", generator_to_id, identity_to_id, DEVICE)
# by_generator = evaluate_by_group(model, test_df, cfg, "generator", generator_to_id, identity_to_id, DEVICE)
# robustness = plot_robustness_curve(model, test_df, cfg, generator_to_id, identity_to_id, DEVICE)


## 12. Inference on a single image

Returns the overall probability plus a per-stream breakdown (forensic score vs.
physiological anomaly score) so you can see *why* the model flagged something — useful
both for debugging and for the "explainability" angle if you go on to write this up.


In [ ]:

@torch.no_grad()
def predict_single_image(model: FADID_PV, image_path: str, cfg: Config, device):
    model.eval()
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)

    face, left_eye, right_eye = align_and_extract(img, cfg.image_size)
    if face is None:
        raise RuntimeError("No face detected.")

    face_float = cv2.cvtColor(face, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    freq_masked = frequency_domain_mask(face_float, "high")  # inspect high-band artifacts at inference
    freq_masked_bgr = cv2.cvtColor((freq_masked * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

    batch = {
        "spatial": to_tensor_norm(face).unsqueeze(0).to(device),
        "freq": to_tensor_norm(freq_masked_bgr).unsqueeze(0).to(device),
        "left_eye": to_tensor_norm(cv2.resize(left_eye, (cfg.eye_crop_size, cfg.eye_crop_size))).unsqueeze(0).to(device),
        "right_eye": to_tensor_norm(cv2.resize(right_eye, (cfg.eye_crop_size, cfg.eye_crop_size))).unsqueeze(0).to(device),
    }
    outputs = model(batch)
    prob_fake = torch.sigmoid(outputs["logits"]).item()
    physio_anomaly = outputs["physio_score"].item()

    return {
        "probability_fake": prob_fake,
        "physiological_anomaly_score": physio_anomaly,
        "verdict": "FAKE" if prob_fake > 0.5 else "REAL",
    }

# Example:
# result = predict_single_image(model, "path/to/test_image.jpg", cfg, DEVICE)
# print(result)


## 13. Structural smoke test (random tensors, no dataset / internet required)

Sanity-checks every module (GRL sign, loss shapes, forward/backward pass) with synthetic
data so you can confirm the pipeline is wired correctly *before* pointing it at a real
multi-gigabyte dataset. Swap `FrozenLNBackbone`'s `timm.create_model(..., pretrained=True)`
for `pretrained=False` here if you're offline.


In [ ]:

def smoke_test(pretrained_backbone: bool = False):
    test_cfg = Config(backbone_name="vit_small_patch16_224", embed_dim=384,
                       backbone_img_size=224, image_size=224, batch_size=4,
                       num_nuisance_classes=3, epochs=1)

    class TinyBackbone(FrozenLNBackbone):
        def __init__(self, model_name, img_size, embed_dim):
            nn.Module.__init__(self)
            self.backbone = timm.create_model(model_name, pretrained=pretrained_backbone, num_classes=0, img_size=img_size)
            self.embed_dim = embed_dim
            self._freeze_all_but_layernorm()

    model = FADID_PV(test_cfg)
    model.spatial_backbone = TinyBackbone(test_cfg.backbone_name, test_cfg.backbone_img_size, test_cfg.embed_dim)
    model.freq_backbone = TinyBackbone(test_cfg.backbone_name, test_cfg.backbone_img_size, test_cfg.embed_dim)
    model.to(DEVICE)

    batch = {
        "spatial": torch.randn(test_cfg.batch_size, 3, 224, 224).to(DEVICE),
        "freq": torch.randn(test_cfg.batch_size, 3, 224, 224).to(DEVICE),
        "left_eye": torch.randn(test_cfg.batch_size, 3, test_cfg.eye_crop_size, test_cfg.eye_crop_size).to(DEVICE),
        "right_eye": torch.randn(test_cfg.batch_size, 3, test_cfg.eye_crop_size, test_cfg.eye_crop_size).to(DEVICE),
        "label": torch.randint(0, 2, (test_cfg.batch_size,)).float().to(DEVICE),
        "generator_id": torch.randint(0, 3, (test_cfg.batch_size,)).to(DEVICE),
        "identity_id": torch.randint(0, 3, (test_cfg.batch_size,)).to(DEVICE),
    }

    optimizer = build_optimizer(model, test_cfg)
    outputs = model(batch)
    losses = composite_loss(outputs, batch, test_cfg)
    losses["total"].backward()
    optimizer.step()

    print("Smoke test passed. Loss breakdown:", {k: round(v.item(), 4) for k, v in losses.items()})
    return model, outputs, losses

# Uncomment to run (needs torch + timm installed; pretrained=False avoids any network call):
# smoke_test(pretrained_backbone=False)


## 14. Anti-overfitting checklist (re-check before you trust any result)

- [ ] Split by `source_video_id`, never by raw image — done in Section 3.
- [ ] Run the **leave-one-generator-out** test (`cfg.holdout_generator`) and report that
      number as prominently as intra-dataset AUC — a 0.98 in-distribution AUC with a 0.65
      zero-day AUC is *not* a generalizing model (this exact failure mode is called out in
      the report's Cross-Dataset Generalization Deficit section).
- [ ] Report AUC **per source dataset** (FF++, Celeb-DF, DFDC, DF-Wild, ...) separately —
      a single pooled number hides which domains you're actually weak on.
- [ ] Report the **robustness curve** (Section 11), not just clean-image AUC.
- [ ] Keep the backbone frozen except LayerNorm; if you ever unfreeze more of it, re-check
      zero-day AUC — full fine-tuning is exactly what the report says causes catastrophic
      forgetting of the foundation model's generalized prior.
- [ ] Balance real:fake and generator-type ratios in each split; a model can "generalize"
      to 90% real-heavy test sets by just predicting real.

## Ethics & scope note

Face imagery and any per-identity metadata are sensitive. Only train/evaluate on datasets
whose license permits your use case (FaceForensics++, Celeb-DF, DFDC, DF-Wild, etc. have
their own terms — read them). Don't run this against images of real people without their
consent outside a licensed research/benchmark context, and don't present a probability
score from a research prototype like this as forensic proof in any legal, workplace, or
accusatory setting — treat it as a research signal, not a verdict.
